# Train bow-card detection and bow-number OCR

This notebook trains both production models on a Colab GPU, validates them, exports ONNX files, checks their runtime tensor contracts, and downloads all results. Before running it, create `build/bow-training-colab.zip` locally with `make colab-all-zip`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%pip install -q -U ultralytics onnx onnxruntime onnxslim opencv-python-headless

In [ ]:
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo
import html
import json
import shutil
import subprocess
import sys
import time
import zipfile

import cv2
import numpy as np
import onnxruntime as ort
import pandas as pd
import torch
import yaml
from google.colab import files
from ultralytics import YOLO

print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU'

## Configuration
The defaults match the local Makefile. Increase epochs or samples here without changing the packaged data.

In [ ]:
CARD_MODEL = 'yolov8s.pt'
CARD_EPOCHS = 100
CARD_PATIENCE = 15
CARD_BATCH = 64
IMAGE_SIZE = 640

OCR_EPOCHS = 30
OCR_PATIENCE = 15
OCR_SAMPLES = 5000
OCR_BATCH = 128
OCR_REAL_FRACTION = 0.75
SEED = 0

## Validate the combined archive

In [ ]:
training_archive = Path('/content/drive/MyDrive/yolo_training/data/bow-training-colab.zip')
assert training_archive.is_file(), f'Missing training archive: {training_archive}'

workspace = Path('/content/bow-model-training')
if workspace.exists():
    shutil.rmtree(workspace)
workspace.mkdir(parents=True)
with zipfile.ZipFile(training_archive) as archive:
    archive.extractall(workspace)

dataset_root = workspace / 'bow-training-data'
card_data = dataset_root / 'card-detector-data'
ocr_data = dataset_root / 'ocr-data'
ocr_script = dataset_root / 'tools' / 'train_bow_crnn.py'
manifest = json.loads((dataset_root / 'manifest.json').read_text())
print(json.dumps(manifest, indent=2))
assert ocr_script.is_file()

card_yaml = card_data / 'data.yaml'
card_config = yaml.safe_load(card_yaml.read_text())
card_config['path'] = str(card_data.resolve())
card_yaml.write_text(yaml.safe_dump(card_config, sort_keys=False))
for split in ('train', 'val'):
    image_count = len(list((card_data / split / 'images').glob('*')))
    label_count = len(list((card_data / split / 'labels').glob('*.txt')))
    print(f'card {split}: {image_count} images, {label_count} labels')
    assert image_count == label_count and image_count > 0
for split in ('train', 'validation'):
    count = len(list((ocr_data / split).glob('*')))
    print(f'OCR {split}: {count} crops')
    assert count > 0

## Train and validate the bow-card detector

In [ ]:
runs = workspace / 'runs'
card_started = time.perf_counter()
card_model = YOLO(CARD_MODEL)
card_model.train(
    data=str(card_yaml), epochs=CARD_EPOCHS, patience=CARD_PATIENCE,
    batch=CARD_BATCH, imgsz=IMAGE_SIZE, device=0, workers=2,
    project=str(runs), name='card-detector', seed=SEED,
    deterministic=True, exist_ok=True,
)
card_minutes = (time.perf_counter() - card_started) / 60
card_best_pt = runs / 'card-detector' / 'weights' / 'best.pt'
card_best = YOLO(str(card_best_pt))
card_validation = card_best.val(data=str(card_yaml), imgsz=IMAGE_SIZE, device=0, plots=True)
card_onnx = Path(card_best.export(format='onnx', imgsz=IMAGE_SIZE, opset=17, simplify=True, dynamic=False))
card_metrics = {
    'train_minutes': card_minutes,
    'precision': float(card_validation.box.mp),
    'recall': float(card_validation.box.mr),
    'mAP50': float(card_validation.box.map50),
    'mAP50-95': float(card_validation.box.map),
}
pd.DataFrame([card_metrics])

## Train and validate whole-card OCR
The packaged training script automatically selects CUDA in Colab and uses held-out real full-string accuracy for checkpoint selection.

In [ ]:
ocr_output = workspace / 'bow_crnn.onnx'
ocr_started = time.perf_counter()
subprocess.check_call([
    sys.executable, str(ocr_script),
    '--epochs', str(OCR_EPOCHS),
    '--patience', str(OCR_PATIENCE),
    '--samples', str(OCR_SAMPLES),
    '--batch-size', str(OCR_BATCH),
    '--real-fraction', str(OCR_REAL_FRACTION),
    '--real-data', str(ocr_data / 'train'),
    '--validation-data', str(ocr_data / 'validation'),
    '--output', str(ocr_output),
    '--keep-checkpoint',
])
ocr_minutes = (time.perf_counter() - ocr_started) / 60
subprocess.check_call([
    sys.executable, str(ocr_script), '--verify', str(ocr_output),
    '--validation-data', str(ocr_data / 'validation'),
    '--min-real-accuracy', '0.85',
])
print(f'OCR training and export: {ocr_minutes:.1f} minutes')

## Verify ONNX contracts and download results

In [ ]:
card_session = ort.InferenceSession(str(card_onnx), providers=['CPUExecutionProvider'])
card_input = card_session.get_inputs()[0]
card_result = card_session.run(None, {card_input.name: np.zeros((1, 3, IMAGE_SIZE, IMAGE_SIZE), np.float32)})[0]
assert card_result.ndim == 3 and card_result.shape[0] == 1 and card_result.shape[1] == 5, card_result.shape

ocr_session = ort.InferenceSession(str(ocr_output), providers=['CPUExecutionProvider'])
ocr_input = ocr_session.get_inputs()[0]
ocr_result = ocr_session.run(None, {ocr_input.name: np.zeros((1, 1, 48, 60), np.float32)})[0]
assert ocr_result.ndim == 3 and ocr_result.shape[1] == 1 and ocr_result.shape[2] == 11, ocr_result.shape

charset = '-0123456789'
def decode_ctc(logits):
    shifted = logits - logits.max(axis=1, keepdims=True)
    probabilities = np.exp(shifted) / np.exp(shifted).sum(axis=1, keepdims=True)
    result, character_confidences, previous = [], [], -1
    for timestep, index in enumerate(logits.argmax(axis=1)):
        if index != previous and index != 0:
            result.append(charset[index])
            character_confidences.append(float(probabilities[timestep, index]))
        previous = index
    confidence = float(np.mean(character_confidences)) if character_confidences else 0.0
    return ''.join(result), confidence

ocr_correct = 0
ocr_total = 0
ocr_failures = []
for path in sorted((ocr_data / 'validation').glob('*.png')):
    expected = path.stem.split('_', 1)[0]
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    image = cv2.resize(image, (60, 48), interpolation=cv2.INTER_AREA)
    tensor = image.astype(np.float32)[None, None] / 255.0
    logits = ocr_session.run(None, {ocr_input.name: tensor})[0][:, 0, :]
    predicted, confidence = decode_ctc(logits)
    if predicted == expected:
        ocr_correct += 1
    else:
        ocr_failures.append({
            'path': path, 'expected': expected, 'predicted': predicted,
            'confidence': confidence,
        })
    ocr_total += 1
ocr_accuracy = ocr_correct / ocr_total
print(f'OCR held-out accuracy: {ocr_correct}/{ocr_total} = {ocr_accuracy:.1%}')
print(f'OCR failures: {len(ocr_failures)}')

failures_dir = workspace / 'ocr-failures'
originals_dir = failures_dir / 'originals'
originals_dir.mkdir(parents=True, exist_ok=True)
report_rows = []
for index, failure in enumerate(ocr_failures, 1):
    source = failure['path']
    predicted_name = failure['predicted'] or 'blank'
    output_name = f"{index:03d}_expected-{failure['expected']}_predicted-{predicted_name}.png"
    original_name = f'{index:03d}_{source.name}'
    shutil.copy2(source, originals_dir / original_name)

    crop = cv2.imread(str(source), cv2.IMREAD_GRAYSCALE)
    enlarged = cv2.resize(crop, (480, 384), interpolation=cv2.INTER_NEAREST)
    enlarged = cv2.cvtColor(enlarged, cv2.COLOR_GRAY2BGR)
    canvas = cv2.copyMakeBorder(enlarged, 58, 4, 4, 4, cv2.BORDER_CONSTANT, value=(20, 20, 20))
    caption = (f"expected {failure['expected']}   predicted {predicted_name}   "
               f"confidence {failure['confidence']:.1%}")
    cv2.putText(canvas, caption, (10, 37), cv2.FONT_HERSHEY_SIMPLEX, 0.62,
                (255, 255, 255), 2, cv2.LINE_AA)
    cv2.imwrite(str(failures_dir / output_name), canvas)
    report_rows.append(
        '<tr>'
        f'<td>{index}</td><td>{html.escape(failure["expected"])}</td>'
        f'<td>{html.escape(predicted_name)}</td><td>{failure["confidence"]:.1%}</td>'
        f'<td><a href="{html.escape(output_name)}"><img src="{html.escape(output_name)}" width="480"></a></td>'
        f'<td><a href="originals/{html.escape(original_name)}">{html.escape(source.name)}</a></td>'
        '</tr>'
    )
report_html = (
    '<!doctype html><html><head><meta charset="utf-8"><title>OCR failures</title>'
    '<style>body{font-family:sans-serif}table{border-collapse:collapse}'
    'th,td{border:1px solid #bbb;padding:6px;vertical-align:top}'
    'th{position:sticky;top:0;background:#eee}img{image-rendering:pixelated}</style>'
    '</head><body>'
    f'<h1>OCR failures: {len(ocr_failures)} of {ocr_total}</h1>'
    '<p>Confidence is the mean selected-character softmax score and is not calibrated.</p>'
    '<table><thead><tr><th>#</th><th>Expected</th><th>Predicted</th>'
    '<th>Confidence</th><th>Enlarged crop</th><th>Original crop</th></tr></thead><tbody>'
    + ''.join(report_rows) + '</tbody></table></body></html>'
)
(failures_dir / 'index.html').write_text(report_html)

summary = {
    'card': card_metrics,
    'ocr': {
        'train_minutes': ocr_minutes,
        'held_out_correct': ocr_correct,
        'held_out_total': ocr_total,
        'held_out_accuracy': ocr_accuracy,
        'failure_count': len(ocr_failures),
    },
    'onnx': {
        'card_input': list(card_input.shape),
        'card_output': list(card_result.shape),
        'ocr_input': list(ocr_input.shape),
        'ocr_output': list(ocr_result.shape),
    },
    'dataset': manifest,
}
summary_path = workspace / 'training-summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

bundle = workspace / 'bow-model-training-results.zip'
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as archive:
    archive.write(card_best_pt, 'card-detector/best.pt')
    archive.write(card_onnx, 'card-detector/bow_card_detect.onnx')
    archive.write(ocr_output, 'ocr/bow_crnn.onnx')
    ocr_checkpoint = ocr_output.with_suffix('.best.pt')
    if ocr_checkpoint.exists():
        archive.write(ocr_checkpoint, 'ocr/bow_crnn.best.pt')
    archive.write(summary_path, summary_path.name)
    results_csv = runs / 'card-detector' / 'results.csv'
    if results_csv.exists():
        archive.write(results_csv, 'card-detector/results.csv')
    for failure_path in sorted(failures_dir.rglob('*')):
        if failure_path.is_file():
            archive.write(failure_path, Path('ocr-failures') / failure_path.relative_to(failures_dir))

timestamp = datetime.now(ZoneInfo('America/Los_Angeles')).strftime('%Y-%m%d_%H%M')
drive_output = Path('/content/drive/MyDrive/yolo_training') / f'bowtrain-{timestamp}'
drive_output.mkdir(parents=True, exist_ok=True)
artifacts = {
    card_best_pt: drive_output / 'bow_card_detect.pt',
    card_onnx: drive_output / 'bow_card_detect.onnx',
    ocr_output: drive_output / 'bow_crnn.onnx',
    summary_path: drive_output / 'training-summary.json',
    bundle: drive_output / bundle.name,
}
if ocr_checkpoint.exists():
    artifacts[ocr_checkpoint] = drive_output / 'bow_crnn.best.pt'
if results_csv.exists():
    artifacts[results_csv] = drive_output / 'card-results.csv'
for source, destination in artifacts.items():
    shutil.copy2(source, destination)
drive_failures = drive_output / 'ocr-failures'
if drive_failures.exists():
    shutil.rmtree(drive_failures)
shutil.copytree(failures_dir, drive_failures)
print(f'Saved training artifacts to {drive_output}')
print('Saved files:', *sorted(path.name for path in drive_output.iterdir()), sep='\n  ')
files.download(str(bundle))